### ファイルインポート

In [107]:
import pandas as pd
import numpy as np
from scipy.stats import skew
import plotly.express as px
import plotly.graph_objects as go

In [108]:
# --- ファイル読み込み ---

df_feats_d = pd.read_parquet("liq_engine_feats_d.parquet", engine="pyarrow")
df_feats_w = pd.read_parquet("liq_engine_feats_w.parquet", engine="pyarrow")

### 負のテールリスク（大損のしやすさ）

In [109]:
df_feats_w.columns

Index(['^GSPC', 'TLT', 'BAMLH0A0HYM2', 'RRPONTSYD', 'DTB3', 'DX-Y.NYB', 'HG=F',
       'GC=F', 'DFII10', 'WALCL', 'WDTGAL', 'WCURCIR', 'NFCI',
       'NDFACBM027SBOG', 'CPF3M', 'RRP_filled', 'Net_Liquidity',
       'Net_Liquidity_z52', 'NFCI_z52', 'Liq_eff', 'Cu_Au_Ratio_z52',
       'NDFACBM027SBOG_z52', 'DFII10_z52', 'CPF3M_DTB3_Spread',
       'CPF3M_DTB3_Spread_z52', 'DXY_z52'],
      dtype='str')

In [110]:
df_feats_w["DXF_roc4"] = df_feats_w["DX-Y.NYB"].pct_change(4)
target_return = 'forward_8w_return'
df_feats_w[target_return] = df_feats_w["^GSPC"].shift(-8) / df_feats_w["^GSPC"] -1

In [111]:
df = df_feats_w.copy()
indicators = ['Liq_eff', 'NFCI_z52', 'DFII10', 'DXF_roc4', 'Cu_Au_Ratio_z52']

for ind in indicators:
    is_inverse = True if ind in ['NFCI', 'DFII10', 'DXF_roc4', 'CP_Spread'] else False
    df[f'{ind}_Regime'] = pd.qcut(df[ind], 5, labels=False, duplicates='drop') + 1
    if is_inverse:
        df[f'{ind}_Regime'] = 6 - df[f'{ind}_Regime'] # 1が最悪(重力)、5が最高(浮力)に統一

stats_list = []
for ind in indicators:
    regime_col = f'{ind}_Regime'
    for r in range(1, 6):
        subset = df[df[regime_col] == r][target_return].dropna()
        # CVaR (期待ショートフォール) : 下位5%の平均リターン
        cvar_5 = subset[subset <= subset.quantile(0.05)].mean()
        stats_list.append({
            'Indicator': ind,
            'Regime': r,
            'Skewness': skew(subset),
            'CVaR_5%': cvar_5,
            'Mean': subset.mean()
        })

stats_df = pd.DataFrame(stats_list)
stats_df

,Indicator,Regime,Skewness,CVaR_5%,Mean
0,Liq_eff,1,-0.476337,-0.118463,0.006235
1,Liq_eff,2,-1.720033,-0.125720,0.010382
2,Liq_eff,3,-0.349848,-0.160895,0.011771
3,Liq_eff,4,-2.177893,-0.172950,0.025878
4,Liq_eff,5,-2.226480,-0.137345,0.018440
5,NFCI_z52,1,-2.287383,-0.132033,0.016095
6,NFCI_z52,2,-1.012540,-0.095630,0.024829
7,NFCI_z52,3,-0.631361,-0.087013,0.023636
8,NFCI_z52,4,-0.717776,-0.197296,0.002156
9,NFCI_z52,5,-0.891945,-0.177916,0.003893


In [112]:
fig = px.bar(
    stats_df[stats_df['Regime'] == 1], 
    x='Indicator', y='CVaR_5%', 
    title="Regime 1 (Heavy Gravity) comparison: CVaR 5% (Bottom Tail Risk)",
    labels={'CVaR_5%': 'Average Return of Bottom 5% Cases'},
    color='Indicator',
    text_auto='.1%'
)
fig.update_layout(yaxis_title="Potential Crash Depth (CVaR)", template="plotly_white")
fig.show(renderer="browser",config=dict(displayModeBar=False))

# 4. 可視化：Skewness（歪度）の比較
fig_skew = px.line(
    stats_df, x='Regime', y='Skewness', color='Indicator', markers=True,
    title="Skewness Transition: From Gravity (1) to Buoyancy (5)",
    labels={'Skewness': 'Distribution Skewness (Negative = Left Tail)'}
)
fig_skew.add_hline(y=0, line_dash="dash")
fig_skew.show(renderer="browser",config=dict(displayModeBar=False))

In [113]:
df_feats_w.columns

Index(['^GSPC', 'TLT', 'BAMLH0A0HYM2', 'RRPONTSYD', 'DTB3', 'DX-Y.NYB', 'HG=F',
       'GC=F', 'DFII10', 'WALCL', 'WDTGAL', 'WCURCIR', 'NFCI',
       'NDFACBM027SBOG', 'CPF3M', 'RRP_filled', 'Net_Liquidity',
       'Net_Liquidity_z52', 'NFCI_z52', 'Liq_eff', 'Cu_Au_Ratio_z52',
       'NDFACBM027SBOG_z52', 'DFII10_z52', 'CPF3M_DTB3_Spread',
       'CPF3M_DTB3_Spread_z52', 'DXY_z52', 'DXF_roc4', 'forward_8w_return'],
      dtype='str')

In [114]:
#df_feats_w = pd.read_parquet("liq_engine_feats.parquet", engine="pyarrow")
df_feats_d["DXY_roc20"] = df_feats_d["DX-Y.NYB"].pct_change(20)
target = "NFCI_z52"
df = df_feats_w[["^GSPC", target]].dropna()
forward_weeks = 40
df['forward_return'] = df['^GSPC'].shift(-forward_weeks) / df['^GSPC'] - 1

# 3. Liq_eff を5つのクオンタイルに分割
# Z-scoreの絶対値で分けるよりも、データ数を揃えた方が統計的な比較がしやすいため
df['Liq_Regime'] = pd.qcut(df[target], 5, 
                          labels=['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)'])

# 4. ナン値を削除（最後の8週間分はリターンが計算できないため）
plot_df = df.dropna(subset=['forward_return', 'Liq_Regime'])

# 5. Plotlyで箱ひげ図を作成
fig = px.box(
    plot_df, 
    x='Liq_Regime', 
    y='forward_return',
    color='Liq_Regime',
    points="all", # 全データ点（ジッター）を表示して密度を確認
    title=f"Market Physics: S&P 500 {forward_weeks}W Forward Return by Liq_eff Regime",
    labels={'forward_return': f'{forward_weeks}-Week Forward Return', 'Liq_Regime': 'Liquidity Regime'},
    category_orders={"Liq_Regime": ['1: Very Low (Heavy)', '2: Low', '3: Neutral', '4: High', '5: Very High (Buoyancy)']}
)

# 0%のライン（損益分岐）を強調
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Break Even")

# レイアウト調整
fig.update_layout(
    xaxis_title="Liquidity Regime (Liq_eff Z-score)",
    yaxis_title=f"Forward {forward_weeks}W Return (%)",
    showlegend=False,
    template="plotly_white",
    yaxis=dict(tickformat=".1%")
)

# 表示
fig.show()

# --- 統計的な裏付け（Skewness）の表示 ---
print("--- Statistical Skewness by Regime ---")
for regime in df['Liq_Regime'].unique():
    if pd.isna(regime): continue
    skew = plot_df[plot_df['Liq_Regime'] == regime]['forward_return'].skew()
    print(f"{regime}: Skewness = {skew:.2f}")

--- Statistical Skewness by Regime ---
4: High: Skewness = -0.64
3: Neutral: Skewness = -0.24
5: Very High (Buoyancy): Skewness = -0.03
2: Low: Skewness = 0.12
1: Very Low (Heavy): Skewness = -0.50
